# Configuración del Cliente en Python
Define las variables de entorno para redirigir el SDK de TypeSafe hacia Ollama (localhost:11434). Ollama no requiere una API Key real, pero el SDK pide pasar un texto cualquiera.

In [1]:
import os
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

os.environ["TYPESAFE_BASE_URL"] = "http://localhost:11434"
os.environ["TYPESAFE_API_KEY"] = "ollama"
os.environ["TYPESAFE_DEFAULT_MODEL"] = "nimble"

## Ejemplo Directo con nimble (Primitivas System One)
- Nimble no genera respuestas largas ni razonamiento en texto
- Responde evaluando directamente las probabilidades de tus preguntas (choice, noul, score):

In [2]:
# Definir el estado (el texto a evaluar) y las preguntas
state_text = "Nuestra pasarela de pago da error 500 desde las 9 AM. Solicitamos reembolso inmediato."

questions = {
    "team": Choice(
        instructions="¿Qué equipo debe atender este caso?",
        criteria={
            "billing": "Cobros, facturas y reembolsos",
            "tech": "Errores de software, caídas y servidores",
            "other": "Otras consultas"
        }
    ),
    "is_urgent": Noul(
        instructions="¿La consulta reporta una falla crítica o urgente?"
    ),
    "urgency_score": Score(
        instructions="Evalúa la prioridad general",
        criteria=["Baja", "Media", "Crítica"]
    )
}

# Ejecutar la consulta en nimble
with TypeSafeClient(timeout=120) as client:
    result = client.system_one(
        state=state_text,
        questions=questions
    )

# Resultados obtenidos en < 100ms
print("Equipo seleccionado:", result.choices["team"].choice)          # 'tech' o 'billing'
print("Es urgente (probabilidad):", result.nouls["is_urgent"].noul)  # e.g., 0.98
print("Score de urgencia (0-2):", result.scores["urgency_score"].score)

Equipo seleccionado: tech
Es urgente (probabilidad): 0.9695285595289549
Score de urgencia (0-2): 1.6122069024535048


## . Integración en Flujos de LangChain (Enrutador Personalizado)
Para integrar nimble como un nodo o middleware de decisión rápido en cadenas o agentes de LangChain (por ejemplo, para elegir qué herramienta o LLM ejecutar sin gastar tokens de un modelo grande):

In [3]:
from langchain_core.runnables import RunnableLambda

def nimble_router_node(inputs: dict) -> str:
    """Nodo que decide el camino a tomar usando nimble."""
    prompt = inputs.get("input", "")
    
    questions = {
        "intent": Choice(
            instructions="Clasifica la intención del usuario.",
            criteria={
                "code": "El usuario pide escribir, depurar o analizar código.",
                "general": "Preguntas de conocimiento general, saludos o conversación.",
                "database": "Consultas sobre tablas, SQL o esquemas."
            }
        )
    }
    
    with TypeSafeClient() as client:
        res = client.system_one(state=prompt, questions=questions)
        
    return res.choices["intent"].choice

# Crear el runnable de LangChain
router_runnable = RunnableLambda(nimble_router_node)

# Probar la decisión
decision = router_runnable.invoke({"input": "Escribe una función en Python para ordenar una lista"})
print("Ruta seleccionada por LangChain:", decision) # Devuelve 'code'

Ruta seleccionada por LangChain: code


In [4]:
# Probar la decisión
decision = router_runnable.invoke({"input": "¿Que hay de postgressql? ¿lo llegaste a instalar?"})
print("Ruta seleccionada por LangChain:", decision) # Devuelve 'code'

Ruta seleccionada por LangChain: database


## Implementación de RAG Ajustado con nimble y LangChain

Un patrón para ajustar un sistema RAG (Retrieval-Augmented Generation) utilizando nimble consiste en emplearlo como un guardrail pre-retrieval y post-retrieval:

- Pre-Retrieval (Evaluación de Necesidad y Enrutamiento): Evalúa si la pregunta realmente requiere buscar en la base de vectores o si puede responderse directamente / es off-topic.

- Retrieval (Búsqueda Vectorial): Recupera los fragmentos (chunks).

- Post-Retrieval (Filtro de Relevancia / Answerability): Verifica si el contexto recuperado realmente contiene la respuesta antes de enviarlo al LLM generador.

In [7]:
from typesafe_sdk import TypeSafeClient, Choice, Noul
from langchain_core.runnables import RunnableLambda

# 1. Enrutador Pre-Retrieval (Decide si buscar o no)
def pre_retrieval_guard(state_input: dict) -> dict:
    query = state_input["query"]
    
    questions = {
        "needs_rag": Noul(
            instructions="¿La pregunta requiere consultar documentación técnica interna o base de conocimientos específica?",
        ),
        "category": Choice(
            instructions="¿A qué área corresponde la duda?",
            criteria={
                "database": "Consultas sobre PostgreSQL, esquemas, SQL o tablas",
                "api": "End points, autenticación, tokens y REST",
                "other": "Preguntas generales o saludos"
            }
        )
    }
    
    with TypeSafeClient() as client:
        res = client.system_one(state=query, questions=questions)
        
    state_input["needs_rag"] = res.nouls["needs_rag"].noul > 0.60
    state_input["category"] = res.choices["category"].choice
    return state_input


# 2. Filtro Post-Retrieval (Verifica si el contexto recuperado es útil)
def post_retrieval_relevance(state_input: dict) -> dict:
    if not state_input.get("needs_rag"):
        return state_input
        
    query = state_input["query"]
    retrieved_chunks = state_input["retrieved_text"]
    
    # Formateamos el estado uniendo la pregunta con los documentos encontrados
    eval_state = f"Pregunta: {query}\n\nDocumentos Recuperados:\n{retrieved_chunks}"
    
    questions = {
        "is_answerable": Noul(
            instructions="¿Los documentos recuperados contienen información suficiente y directa para responder a la pregunta del usuario?"
        )
    }
    
    with TypeSafeClient() as client:
        res = client.system_one(state=eval_state, questions=questions)
        
    # Asignamos el flag de validez
    state_input["is_relevant"] = res.nouls["is_answerable"].noul >= 0.70
    return state_input


# Construcción del Flujo en LangChain
rag_pre_check = RunnableLambda(pre_retrieval_guard)
rag_post_check = RunnableLambda(post_retrieval_relevance)

# Simulación de ejecución
input_data = {"query": "¿Cómo configuro el pool de conexiones en PostgreSQL?"}
print(input_data)

# Paso 1: Evaluación Previa
step1 = rag_pre_check.invoke(input_data)
print("¿Requiere RAG?:", step1["needs_rag"]) # True
print("Categoría:", step1["category"])        # 'database'

# Simulación de búsqueda en Vector DB
step1["retrieved_text"] = "Para configurar pg_isready o max_connections en postgresql.conf..."

# Paso 2: Evaluación Posterior a la búsqueda
step2 = rag_post_check.invoke(step1)
print("¿El contexto recuperado sirve?:", step2.get("is_relevant", "N/A (No se requirió RAG)"))

{'query': '¿Cómo configuro el pool de conexiones en PostgreSQL?'}
¿Requiere RAG?: False
Categoría: database
¿El contexto recuperado sirve?: N/A (No se requirió RAG)


In [8]:
# Simulación de ejecución
input_data = {"query": "si estoy enfermo debo ir al medico?"}
print(input_data, "\n\n\n")

# Paso 1: Evaluación Previa
step1 = rag_pre_check.invoke(input_data)
print("¿Requiere RAG?:", step1["needs_rag"]) # True
print("Categoría:", step1["category"])        # 'database'

# Simulación de búsqueda en Vector DB
step1["retrieved_text"] = "Para configurar pg_isready o max_connections en postgresql.conf..."

# Paso 2: Evaluación Posterior a la búsqueda
step2 = rag_post_check.invoke(step1)
print("¿El contexto recuperado sirve?:", step2.get("is_relevant", "N/A (No se requirió RAG)"))

{'query': 'si estoy enfermo debo ir al medico?'} 



¿Requiere RAG?: False
Categoría: other
¿El contexto recuperado sirve?: N/A (No se requirió RAG)
